# DL Model Notebook: LSTM Forecasting Pipeline

This notebook trains a sequence model for next-day close forecasting, evaluates it, and exports backend-ready artifacts.

Target artifacts:
- `lstm_model.keras`
- `close_scaler.pkl`

In [ ]:
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'backend').exists() and (REPO_ROOT.parent / 'backend').exists():
    REPO_ROOT = REPO_ROOT.parent

RAW_DIR = REPO_ROOT / 'data' / 'raw'
MODEL_DIR = REPO_ROOT / 'backend' / 'models'
RAW_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def load_series(symbol='AAPL', period='5y'):
    csv_path = RAW_DIR / f'{symbol.lower()}_dl_raw.csv'
    if csv_path.exists():
        frame = pd.read_csv(csv_path)
        frame['date'] = pd.to_datetime(frame['date'])
        return frame

    try:
        frame = yf.download(symbol, period=period, auto_adjust=False, progress=False).reset_index()
        frame.columns = [c.lower().replace(' ', '_') for c in frame.columns]
        frame = frame[['date', 'open', 'high', 'low', 'close', 'volume']]
        frame.to_csv(csv_path, index=False)
        return frame
    except Exception:
        dates = pd.date_range(end=pd.Timestamp.today(), periods=700, freq='B')
        rng = np.random.default_rng(7)
        base = np.linspace(80, 200, len(dates)) + rng.normal(0, 2, len(dates)).cumsum()
        frame = pd.DataFrame({
            'date': dates,
            'open': base + rng.normal(0, 1.5, len(dates)),
            'high': base + rng.normal(2, 1.5, len(dates)),
            'low': base - rng.normal(2, 1.5, len(dates)),
            'close': base,
            'volume': rng.integers(800000, 2500000, len(dates)),
        })
        frame.to_csv(csv_path, index=False)
        return frame

In [ ]:
df = load_series('AAPL')
df = df.drop_duplicates().sort_values('date').reset_index(drop=True)
df[['open', 'high', 'low', 'close', 'volume']] = df[['open', 'high', 'low', 'close', 'volume']].apply(pd.to_numeric, errors='coerce')
df = df.fillna(method='ffill').fillna(method='bfill')
df['return_1'] = df['close'].pct_change().fillna(0)
df['sma_10'] = df['close'].rolling(10).mean()
df['sma_20'] = df['close'].rolling(20).mean()
df['momentum_5'] = df['close'].diff(5)
df['volatility_10'] = df['return_1'].rolling(10).std().fillna(0)
df = df.dropna().reset_index(drop=True)
display(df.head())
display(df.describe().transpose())

In [ ]:
sequence_features = ['close', 'volume', 'return_1', 'sma_10', 'sma_20', 'momentum_5', 'volatility_10']
sequence_length = 60
feature_frame = df[sequence_features].copy()
scaler = MinMaxScaler()
scaled = scaler.fit_transform(feature_frame)

def build_sequences(array, window_size=60):
    X, y = [], []
    for index in range(window_size, len(array)):
        X.append(array[index - window_size:index])
        y.append(array[index, 0])
    return np.array(X), np.array(y)

X, y = build_sequences(scaled, sequence_length)
split_index = int(len(X) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1),
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', patience=5, factor=0.5, min_lr=1e-5),
]
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
pred_scaled = model.predict(X_test, verbose=0)
close_min = scaler.data_min_[0]
close_max = scaler.data_max_[0]

y_test_actual = y_test * (close_max - close_min) + close_min
pred_actual = pred_scaled.flatten() * (close_max - close_min) + close_min

rmse = np.sqrt(mean_squared_error(y_test_actual, pred_actual))
mae = mean_absolute_error(y_test_actual, pred_actual)
print({'rmse': rmse, 'mae': mae})

plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Training History')
plt.legend()
plt.tight_layout()
plt.show()

comparison = pd.DataFrame({
    'actual_close': y_test_actual,
    'predicted_close': pred_actual,
})
comparison.head(20)

comparison.plot(title='Actual vs Predicted Close Price')
plt.tight_layout()
plt.show()

In [ ]:
model_path = MODEL_DIR / 'lstm_model.keras'
scaler_path = MODEL_DIR / 'close_scaler.pkl'
model.save(model_path)
joblib.dump(scaler, scaler_path)
print('Saved:', model_path)
print('Saved:', scaler_path)

latest_window = X_test[-1:]
next_prediction = model.predict(latest_window, verbose=0)[0, 0]
print('Scaled next prediction:', next_prediction)

## Backend Integration Notes

The backend forecast service looks for the exported Keras model and scaler in `backend/models/`. This notebook keeps all training logic local and only emits artifacts for API inference.